In [2]:
import pandas as pd
import re
from konlpy.tag import Okt
from tqdm import tqdm

# 1. 데이터 로드
df = pd.read_csv('jeju_travel_최종본.csv')

# 2. 텍스트 정제 함수
def clean_text(text):
    if pd.isna(text): return ""
    # 한글과 공백만 남기고 제거
    text = re.sub(r'[^가-힣\s]', ' ', text)
    # 중복 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 3. 명사 추출 및 불용어 제거 함수
okt = Okt()
stop_words = ['제주', '제주도', '정말', '너무', '진짜', '방문', '사람', '곳', '것', '여기', '다시', '그냥', '우리', '정도']

def get_keywords(text):
    cleaned = clean_text(text)
    nouns = okt.nouns(cleaned)
    # 2글자 이상인 명사만 선택 + 불용어 제외
    return " ".join([w for w in nouns if len(w) > 1 and w not in stop_words])

# 4. 전체 데이터 적용 (tqdm으로 진행 상황 보기)
print("데이터 정제 및 키워드 추출 시작...")
tqdm.pandas()
df['keywords'] = df['reviews_text'].progress_apply(get_keywords)

# 5. 최종 저장
df.to_csv('jeju_cleaned_data.csv', index=False, encoding='utf-8-sig')
print("정리 끝! 'jeju_cleaned_data.csv' 파일이 생성되었습니다.")

데이터 정제 및 키워드 추출 시작...


100%|██████████| 1057/1057 [01:41<00:00, 10.44it/s]

정리 끝! 'jeju_cleaned_data.csv' 파일이 생성되었습니다.


In [2]:
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv('jeju_cleaned_1차.csv')

# 2. 더 강력해진 카페 판별 키워드
cafe_strong_indicators = [
    '카페', '커피', '디저트', '베이커리', '소금빵', '케이크', '스콘', 
    '휘낭시에', '라떼', '아메리카노', '핸드드립', '로스팅', '타르트', 
    '에이드', '차', '티룸', '구움과자', '젤라또', '푸딩'
]

def ultimate_cafe_finder(row):
    # 대상: 맛집, 기타, 문화, 예술 등 '자연'을 제외한 모든 카테고리 검사
    if row['category'] in ['맛집', '기타', '문화', '관광지']: 
        name_val = str(row['name'])
        kw_val = str(row['keywords'])
        
        # 조건 1: 이름에 카페 관련 핵심어 포함
        if any(x in name_val for x in ['카페', '커피', '베이커리', '로스터리', '빵집', '디저트']):
            return '카페'
        
        # 조건 2: 리뷰 키워드에서 카페 지표가 3개 이상 발견됨
        match_count = sum(1 for word in cafe_strong_indicators if word in kw_val)
        if match_count >= 3:
            return '카페'
            
    return row['category']

# 3. 전체 적용
df['category'] = df.apply(ultimate_cafe_finder, axis=1)

# 4. 결과 확인
print("--- 최종 카테고리 분포 ---")
print(df['category'].value_counts())

# 5. 저장
df.to_csv('jeju_total_refined_v3.csv', index=False, encoding='utf-8-sig')

--- 최종 카테고리 분포 ---
category
맛집    365
카페    213
자연    205
문화    189
기타     85
Name: count, dtype: int64


In [3]:
import pandas as pd

# 1. 최신 데이터 로드
df = pd.read_csv('jeju_total_refined_v3.csv')

# 2. 수동 수정 리스트
manual_fixes = {
    '경미네집': '맛집',
    '글라글라하와이': '맛집',
    '뿌리와열매': '맛집',
    '넥슨컴퓨터박물관': '문화',
    '방주교회': '기타'
}

for name, cat in manual_fixes.items():
    df.loc[df['name'] == name, 'category'] = cat

# 3. 텍스트 기반 자동 교정 (식사 키워드가 너무 강한 카페들)
meal_words = ['해물라면', '전복죽', '성게', '뚝배기', '흑돼지', '국수', '조림', '구이', '비빔밥']
cafe_words = ['케이크', '디저트', '베이커리', '스콘', '휘낭시에', '라떼']

def final_check(row):
    if row['category'] == '카페':
        kw = str(row['keywords'])
        meal_count = sum(1 for w in meal_words if w in kw)
        cafe_count = sum(1 for w in cafe_words if w in kw)
        
        # 식사 단어가 2개 이상이고 디저트 단어가 거의 없으면 맛집으로 판정
        if meal_count >= 2 and cafe_count <= 1:
            return '맛집'
    return row['category']

df['category'] = df.apply(final_check, axis=1)

# 4. 최종 저장
df.to_csv('jeju_final_fixed_v4.csv', index=False, encoding='utf-8-sig')

print("✅ '경미네집' 포함 카테고리 오분류 수정 완료!")
print(f"✅ 최종 저장 파일명: jeju_final_fixed_v4.csv")

✅ '경미네집' 포함 카테고리 오분류 수정 완료!
✅ 최종 저장 파일명: jeju_final_fixed_v4.csv
